In [ ]:
%matplotlib ipympl
%matplotlib inline
%load_ext autoreload
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import pandas as pd
import os
import json
from PIL import Image
import re
import torch.nn as nn
import seaborn as sns
import random
import numpy as np
from scipy.stats import pearsonr
import torch
from sklearn.metrics.pairwise import cosine_similarity
import time
import sys
import torchvision.transforms as transforms
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModel, ModernBertConfig, ModernBertModel
from IPython.core.display_functions import clear_output
sys.path.append('/Users/orenm/Desktop/code_projects/BlenderShaderProject/project_files/')

In [ ]:
from Logic.blender_tree_manager import BlenderTreeManager
from Logic.tree_networks_manager import TreesNetworkManager
from Logic.utils import custom_shortest_paths, get_example_edges, are_identical_images, lc

In [ ]:
path = '/Users/orenm/BlenderShaderProject/data/'
images_path = os.path.join(path, 'images/')
db_path = os.path.join(path, 'DB/')
dataset_path = os.path.join(path, "datasets/")
active_models_path = os.path.join(path, "active_models/")
code_strings_file = os.path.join(dataset_path, "network_managers_strings.json")
tokenizer_path = os.path.join(active_models_path, "my_tokenizer")

In [ ]:
db_manager = TreesNetworkManager.load(db_path)
len(db_manager.network)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)

In [ ]:
def get_example_edges(
    trees_network_manager: TreesNetworkManager, n_examples: int, target_distance=None, target_node_label=None, from_node_label=None, max_tries=10
):
    """
    Sample edges from the tree network manager such that each edge leads closer to a target node
    """
    if from_node_label is not None:
        from_nodes = list(trees_network_manager.get_nodes_with_label(from_node_label))
    else:
        from_nodes = list(trees_network_manager.network.nodes)

    edges_selected = defaultdict(set)
    for i in lc(range(n_examples), print_every=50):
        for _ in range(max_tries):
            from_node = np.random.choice(from_nodes)
            distance, target_node, neighbor = find_edge_for_target_label(
                trees_network_manager, from_node, target_label=target_node_label, target_distance=target_distance
            )

            if target_node is not None and (from_node, neighbor, target_node) not in edges_selected[distance]:
                edges_selected[distance].add((from_node, neighbor, target_node))
                break
    return edges_selected

# plot regular distances

In [ ]:
all_distances = []
for i in lc(range(3000), print_every=50):
    node1 = np.random.choice(db_manager.network.nodes)
    # Get all nodes at distance k from the current node
    distances = custom_shortest_paths(db_manager.network, node1, cutoff=30)
    all_distances.extend(distances.values())

In [ ]:
distance_distribution = pd.Series(all_distances).value_counts(normalize=True)

In [ ]:
distance_distribution.sort_index().plot()

In [ ]:
# only cluster base
all_distances = []
target_nodes = db_manager.get_nodes_with_label(IS_CLUSTER_BASE)
for node in lc(target_nodes, print_every=50):
    distances = custom_shortest_paths(db_manager.network, node, cutoff=30)
    all_distances.extend(distances.values())

In [ ]:
pd.Series(all_distances).value_counts(normalize=True).sort_index().plot()

In [ ]:
edges_dist = pd.Series([x['variation_type'] for _, _, x in db_manager.network.edges(data=True)]).value_counts()
edges_dist

# make dataset

Sampling random edges from the huge network of trees we built.
To make sure there is variance, we sample specifically trees with special labels like empty trees (called empty networks).
This is important so the dataset will include examples where the start position is an empty...

In [ ]:
def update_sets_in_dict(d, to_update):
    for key in to_update:
        d[key].update(to_update[key])

edges_by_distance = defaultdict(set)

In [ ]:
max_distance = 24
min_distance = 1

update_sets_in_dict(edges_by_distance, get_example_edges(db_manager, n_examples=150000,
                                                         target_node_label=ON_PATH_TO_EMPTY, from_node_label=IS_EMPTY_NETWORK))

update_sets_in_dict(edges_by_distance, get_example_edges(db_manager, n_examples=150000,
                                                        target_node_label=ON_PATH_TO_EMPTY, from_node_label=ON_PATH_TO_EMPTY))

update_sets_in_dict(edges_by_distance, get_example_edges(db_manager, n_examples=100000,
                                                         target_node_label=IS_CLUSTER_BASE, from_node_label=ON_PATH_TO_EMPTY))

update_sets_in_dict(edges_by_distance, get_example_edges(db_manager, n_examples=60000, target_node_label=IS_CLUSTER_BASE))
# update_sets_in_dict(edges_by_distance, get_example_edges(db_manager, n_examples=150000))

# for distance in lc(range(min_distance, max_distance+1)):
#     update_sets_in_dict(edges_by_distance, get_example_edges(db_manager, n_examples=5000, target_distance=distance,
#                                                          target_node_label=None, from_node_label=None))

In [ ]:
# just a sanity check - the edges are correct
for distance, examples in edges_by_distance.items():
    for from_node, neighbor, target_node in random.sample(sorted(examples), min(1000, len(examples))):
        assert nx.shortest_path_length(db_manager.network, source=neighbor, target=target_node, weight=edge_weight_function) == distance - 1
        assert nx.shortest_path_length(db_manager.network, source=from_node, target=target_node, weight=edge_weight_function) == distance
        assert db_manager.network.has_edge(from_node, neighbor)

In [ ]:
# how many did we really get from an empty network
label_counts = {}
for key, values in edges_by_distance.items():
    label_counts[key] = sum([db_manager.node_has_label(target_node, IS_CLUSTER_BASE) for from_node, neighbor, target_node in values])
counts = pd.Series({key: len(values) for key, values in edges_by_distance.items()}).sort_index()
counts.plot(label='all edges')
pd.Series(label_counts).sort_index().plot(label='with label')
plt.legend()

In [ ]:
counts.sum(), pd.Series(label_counts).sum()

# filter some of the results

In [ ]:
# filter if target and from node are similar or if target is empty network or image
deleted = 0
variation_types_by_distance = {}
for key in lc(sorted(list(edges_by_distance))):
    variation_types = defaultdict(list)
    for from_node, neighbor, target_node in list(edges_by_distance[key]):
        path1 = db_manager.make_image_path(from_node, images_path)
        path2 = db_manager.make_image_path(target_node, images_path)
        if db_manager.node_has_label(target_node, IS_EMPTY_IMAGE) or db_manager.node_has_label(target_node, IS_EMPTY_NETWORK):
            deleted+=1
            continue
        elif key < 3 and are_identical_images(path1, path2):  # for longer distances really unlikely they'll be the same so saves calculation
            deleted+=1
            continue
        else:
            edge_data = db_manager.network[from_node][neighbor]
            variation_types[edge_data['variation_type']].append((from_node, neighbor, target_node))
    variation_types_by_distance[key] = variation_types
print(f'deleted {deleted}')

In [ ]:
counts = {}
for key, edges_by_type in list(variation_types_by_distance.items()):
    counts[key] = pd.Series({key: len(value) for key, value in  edges_by_type.items()})
counts_type_by_distance = pd.DataFrame(counts).sort_index(axis=1).fillna(0).astype(int)
counts_type_by_distance

In [ ]:
counts_type_by_distance.T.plot()

In [ ]:
counts_type_by_distance.sum(axis=1) / counts_type_by_distance.sum(axis=1).sum()

In [ ]:
# sample to have fewer of some types of edges - no more than 20% of each type (numeric can be too much)
for key, edges_by_type in list(variation_types_by_distance.items()):
    total = sum([len(v) for v in edges_by_type.values()])
    for action, edges_for_action in list(edges_by_type.items()):
        edges_by_type[action] = random.sample(edges_for_action, min(len(edges_for_action), int(total/5)))

In [ ]:
counts = {}
for key, edges_by_type in list(variation_types_by_distance.items()):
    counts[key] = pd.Series({key: len(value) for key, value in  edges_by_type.items()})
counts_type_by_distance = pd.DataFrame(counts).sort_index(axis=1).fillna(0).astype(int)
counts_type_by_distance

In [ ]:
counts_type_by_distance.sum(axis=1)

In [ ]:
pd.Series({key: c.sum() for key, c in counts.items()}).sort_index().plot()

In [ ]:
counts_type_by_distance.sum().sum()

# add 0s - self reference and do not change anything...

In [ ]:
n_self_references = 50000
self_nodes = random.sample(list(db_manager.network.nodes), n_self_references)
variation_types_by_distance[0] = {'something': [(node, None, node) for node in self_nodes]}

In [ ]:
# !!! need to filter out empty images and empty networks!!!!

In [ ]:
with open(os.path.join(path, 'tmp_edges_by_distance.json'), 'w') as f:
    json.dump(variation_types_by_distance, f)

# make into edge examples datasets

In [ ]:
all_edge_data = []
for distance, edges_type in lc(variation_types_by_distance.items()):
    for edges in edges_type.values():
        for from_node, neighbor, target_node in edges:
            edge_data = db_manager.network[from_node][neighbor] if neighbor else None
            # randomly for some examples we add the "source" image
            # later we can test how important it is for the model to see what the current tree makes...
            # if it's not important - then it saves time on generating those images
            add_cur_image = np.random.rand() < 0.5
            code = db_manager.network_managers[from_node].to_str(with_seeds=False, add_image_tokens=True, with_cur_image = add_cur_image)
            example_data = edge_to_example(edge_data, code, tokenizer)
            if example_data is None:
                continue
            assert all(example_data['attention_mask'])  # should only be ones, right?
            del example_data['attention_mask']
            all_edge_data.append((from_node, target_node, add_cur_image, distance, example_data))

In [ ]:
# just a sanity check - the edges are correct
for from_node, target_node, add_cur_image, distance, example_data in random.sample(all_edge_data, 1000):
    assert nx.shortest_path_length(db_manager.network, source=from_node, target=target_node, weight=edge_weight_function) == distance

In [ ]:
len(all_edge_data)

# separate clusters to test and train

In [ ]:
connected_components = list(nx.strongly_connected_components(db_manager.network))

In [ ]:
test_size = 0.05

random.shuffle(connected_components)
n_clusters = len(connected_components)
n_test_clusters = int((n_clusters * test_size))
n_test_clusters

In [ ]:
test_clusters = connected_components[:n_test_clusters]
train_clusters = connected_components[n_test_clusters:]
test_nodes = {node for cluster in test_clusters for node in cluster}
train_nodes = {node for cluster in train_clusters for node in cluster}

In [ ]:
len(test_nodes), len(train_nodes)

In [ ]:
train_data = []
test_data = []
for example in all_edge_data:
    if example[0] in train_nodes:
        train_data.append(example)
    elif example[0] in test_nodes:
        test_data.append(example)
    else:
        raise

In [ ]:
len(train_data), len(test_data)

In [ ]:
with open(os.path.join(dataset_path, 'train_dataset_for_corrector.json'), "w") as f:
    json.dump(train_data, f)

with open(os.path.join(dataset_path, 'test_dataset_for_corrector.json'), "w") as f:
    json.dump(test_data, f)